# QBO time–height plot: ne30pg3 (SE physgrid) CAM vs. ERA5

Builds the classic QBO diagnostic — a time–height section of equatorial zonal-mean
zonal wind (`U`) — from monthly-mean CAM history (`h0a`) files, **and** compares it
against an ERA5 reanalysis climatology regridded to the same L93 vertical levels.

**The model hitch**: the CAM files are on the unstructured `ne30pg3` spectral-element
physics grid (dimension `ncol`, no regular lat/lon grid to average over). There is
no "average over longitude at fixed latitude" the way there would be on a regular
grid. Instead, this notebook approximates the zonal mean by taking an
**area-weighted average over all grid columns within an equatorial latitude band**
(default ±5°), using each column's `area` field as the weight. Since the columns
in that band are spread across all longitudes, this is a good approximation to a
true zonal mean.

**ERA5, by contrast**, has already been regridded to a regular longitude ×
(near-)Gaussian-latitude grid (288 × 192, ~1.25° lon spacing), so its zonal mean is
the ordinary textbook definition: a plain average over longitude, then a
`cos(lat)`-weighted average over the same equatorial latitude band — no
unstructured-grid bookkeeping needed. Conveniently, this ERA5 product has also been
regridded onto CAM's own L93 hybrid vertical levels, so it can share the model's
pressure axis exactly.

Data sources:
- Model: `/glade/derecho/scratch/juliob/archive/NewMMsrc_fmt_c64173-x11/atm/hist`
- ERA5: `/glade/campaign/cgd/amp/juliob/ERA5/fv1x1_L93_monthly/ZYX`

In [ ]:
import glob
import re
from datetime import datetime

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import netCDF4 as nc
import numpy as np

In [ ]:
# ----------------------------- Configuration -----------------------------
CASE = "NewMMsrc_fmt_c64173-x11"
CASE = "PySRsrc_fmt_c64173-x02"
ARCHDIR = f"/glade/derecho/scratch/juliob/archive/{CASE}/atm/hist"
FILE_GLOB = f"{ARCHDIR}/{CASE}.cam.h0a.*-*.nc"

ERA5_DIR = "/glade/campaign/cgd/amp/juliob/ERA5/fv1x1_L93_monthly/ZYX"
ERA5_GLOB = f"{ERA5_DIR}/ERA5.fv1x1_L93.time.*-*.nc"

LAT_BAND = 5.0          # equatorial band, |lat| <= LAT_BAND degrees
P0 = 100000.0           # Pa; CAM's standard reference pressure (not stored in these files)

PLOT_YLIM_HPA = (100.0, 1.0)   # standard QBO pressure range (high pressure at bottom)
OUT_PNG = "qbo_ne30pg3_timeheight.png"

## 1. Locate files and build the equatorial-band mask + area weights

The grid (`lat`, `lon`, `area`) is static across the run, so it only needs to be
read once, from the first file.

In [ ]:
files = sorted(glob.glob(FILE_GLOB))
print(len(files), "monthly files found")
assert len(files) > 0, "No files matched — check ARCHDIR/FILE_GLOB above."

with nc.Dataset(files[0]) as f0:
    lat = f0.variables["lat"][:]
    area = f0.variables["area"][:]
    hyam = f0.variables["hyam"][:]
    hybm = f0.variables["hybm"][:]
    lev = f0.variables["lev"][:]           # nominal pressure levels [hPa]
    nlev = f0.dimensions["lev"].size

eq_mask = np.abs(lat) <= LAT_BAND
weights = area[eq_mask] / area[eq_mask].sum()
print(f"{eq_mask.sum()} of {lat.size} columns fall within |lat| <= {LAT_BAND} deg")

## 2. Loop over months, building the area-weighted equatorial-mean U(lev) time series

**Performance note**: reading `U[0, :, eq_mask]` directly with a boolean/fancy
column index is slow (the ~4000 equatorial columns are scattered, not contiguous,
among the 48600 total columns — netCDF/HDF5 has to fetch them essentially one at a
time). It is far faster to read the full `(nlev, ncol)` slab for each month
(a single contiguous read) and then subset/average in memory with numpy. That
difference is roughly 6x per file, and matters a lot over hundreds of months.

In [ ]:
date_re = re.compile(r"\.h0a\.(\d{4})-(\d{2})\.nc$")

dates = []
U_th = np.full((len(files), nlev), np.nan)     # (time, lev)
PS_eq = np.full(len(files), np.nan)            # equatorial-mean surface pressure [Pa]

for i, fp in enumerate(files):
    m = date_re.search(fp)
    yyyy, mm = int(m.group(1)), int(m.group(2))
    dates.append(datetime(yyyy, mm, 15))

    with nc.Dataset(fp) as f:
        U_full = f.variables["U"][0, :, :]     # (nlev, ncol) - contiguous read
        PS_full = f.variables["PS"][0, :]      # (ncol,)      - contiguous read

    U_eq = U_full[:, eq_mask]                  # subset in memory (fast)
    PS_eq_col = PS_full[eq_mask]

    U_th[i, :] = np.sum(U_eq * weights[None, :], axis=1)
    PS_eq[i] = np.sum(PS_eq_col * weights)

print("done:", U_th.shape, "(time, lev)")

## 3. Sanity check: nominal `lev` vs. the actual local equatorial-mean pressure

CAM's hybrid coordinate gives the true local pressure as
`p = hyam*P0 + hybm*PS`. In the stratosphere (where the QBO lives), `hybm` is
essentially zero, so the nominal `lev` value is already an excellent stand-in for
the actual pressure — which is what makes it usable as a fixed y-axis for a
contour plot in the first place (the real local pressure varies slightly in time
with `PS`, which a contour plot's fixed grid can't represent anyway).

In [ ]:
p_actual_hPa = (hyam[None, :] * P0 + hybm[None, :] * PS_eq[:, None]) / 100.0
strat = lev <= 100.0
print(
    "max |actual eq-mean pressure - nominal lev| for lev <= 100 hPa:",
    f"{np.nanmax(np.abs(p_actual_hPa[:, strat] - lev[None, strat])):.4f} hPa",
)

## 3b. ERA5 comparison: build the same equatorial-band time–height series

Same target quantity (equatorial-band, pressure-level zonal-mean `U`), but on a
regular grid, so the averaging is the standard two-step zonal mean: a plain
longitude average (the grid spacing is uniform in longitude, so this is just
`.mean(axis=lon)`), followed by a `cos(lat)`-weighted average over the retained
latitude rows in the equatorial band.

Same performance consideration as the model loop: a full `(lev, lat, lon)` read
per month is faster than a fancy-indexed latitude-band read, so that's what's
used here too, even though the equatorial band is only 10 contiguous latitude
rows out of 192.

In [ ]:
era5_files = sorted(glob.glob(ERA5_GLOB))
print(len(era5_files), "ERA5 monthly files found")
assert len(era5_files) > 0, "No ERA5 files matched — check ERA5_DIR/ERA5_GLOB above."

with nc.Dataset(era5_files[0]) as f0:
    lat_e = f0.variables["lat"][:]
    lev_e = f0.variables["lev"][:]

assert np.allclose(lev_e, lev), "ERA5 and model lev grids differ — cannot share one y-axis."

lat_mask_e = np.abs(lat_e) <= LAT_BAND
lat_weights_e = np.cos(np.deg2rad(lat_e[lat_mask_e]))
lat_weights_e = lat_weights_e / lat_weights_e.sum()
print(f"{lat_mask_e.sum()} of {lat_e.size} ERA5 latitude rows fall within |lat| <= {LAT_BAND} deg")

date_re_era5 = re.compile(r"\.time\.(\d{4})-(\d{2})\.nc$")

dates_era5 = []
U_th_era5 = np.full((len(era5_files), len(lev_e)), np.nan)

for i, fp in enumerate(era5_files):
    m = date_re_era5.search(fp)
    yyyy, mm = int(m.group(1)), int(m.group(2))
    dates_era5.append(datetime(yyyy, mm, 15))

    with nc.Dataset(fp) as f:
        U_full = f.variables["U"][0, :, :, :]      # (lev, lat, lon) - contiguous read

    U_band = U_full[:, lat_mask_e, :]               # subset in memory (fast)
    U_zonal = U_band.mean(axis=2)                   # plain longitude average
    U_th_era5[i, :] = np.sum(U_zonal * lat_weights_e[None, :], axis=1)

print("done:", U_th_era5.shape, "(time, lev)")

## 4. Plot: time–height section

Standard QBO-plot conventions used here:
- **log-pressure y-axis**, inverted (higher pressure/lower altitude at the bottom)
- **diverging colormap centered at zero** (blue = easterly, red = westerly — the
  meteorological convention), so the sign of the wind (the actual QBO signal) reads
  immediately rather than being buried in a sequential/rainbow scale
- a highlighted **zero-wind contour line** marking the phase boundary between
  easterly and westerly regimes

In [ ]:
vmax = 40.0  # capped to the QBO layer; contourf's extend='both' clips/saturates beyond this
levels = np.arange(-vmax, vmax + 1e-6, 5.0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

for ax, data, dts, label in (
    (ax1, U_th, dates, f"CAM ({CASE})"),
    (ax2, U_th_era5, dates_era5, "ERA5"),
):
    cf = ax.contourf(dts, lev, data.T, levels=levels, cmap="RdBu_r", extend="both")
    ax.contour(dts, lev, data.T, levels=[0.0], colors="0.15", linewidths=1.2)
    ax.set_yscale("log")
    ax.set_ylim(PLOT_YLIM_HPA)
    ax.set_yticks([100, 70, 50, 30, 20, 10, 7, 5, 3, 2, 1])
    ax.set_yticklabels(["100", "70", "50", "30", "20", "10", "7", "5", "3", "2", "1"])
    ax.set_ylabel("Pressure (hPa)")
    ax.set_title(f"Equatorial zonal-mean zonal wind (|lat| \u2264 {LAT_BAND:g}\u00b0) \u2014 {label}")

ax2.xaxis.set_major_locator(mdates.YearLocator(2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.set_xlabel("Year")

cbar = fig.colorbar(cf, ax=[ax1, ax2], pad=0.02)
cbar.set_label("U (m s$^{-1}$)")

fig.savefig(OUT_PNG, dpi=150)
print("saved", OUT_PNG)

In [ ]:
vmax = 40.0  # capped to the QBO layer; contourf's extend='both' clips/saturates beyond this
levels = np.arange(-vmax, vmax + 1e-6, 5.0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

for ax, data, dts, label in (
    (ax1, U_th, dates, f"CAM ({CASE})"),
    (ax2, U_th, dates, "--"),
):
    cf = ax.contourf(dts, lev, data.T, levels=levels, cmap="RdBu_r", extend="both")
    ax.contour(dts, lev, data.T, levels=[0.0], colors="0.15", linewidths=1.2)
    ax.set_yscale("log")
    ax.set_ylim(PLOT_YLIM_HPA)
    #ax.set_xlim( 0, 20*12)
    ax.set_yticks([100, 70, 50, 30, 20, 10, 7, 5, 3, 2, 1])
    ax.set_yticklabels(["100", "70", "50", "30", "20", "10", "7", "5", "3", "2", "1"])
    ax.set_ylabel("Pressure (hPa)")
    ax.set_title(f"Equatorial zonal-mean zonal wind (|lat| \u2264 {LAT_BAND:g}\u00b0) \u2014 {label}")

#ax2.xaxis.set_major_locator(mdates.YearLocator(2))
#ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
#ax2.set_xlabel("Year")

cbar = fig.colorbar(cf, ax=[ax1, ax2], pad=0.02)
cbar.set_label("U (m s$^{-1}$)")

fig.savefig(OUT_PNG, dpi=150)
print("saved", OUT_PNG)

In [ ]:
print( dts[0] )

## 5. (Optional) QBO index at a single level: CAM vs. ERA5

A simple 1-D companion time series — equatorial-mean U at a chosen level (default
≈10 hPa, a standard QBO reference level) — useful for reading off period and
amplitude directly, with both records overlaid.

In [ ]:
target_hpa = 10.0
ilev = int(np.argmin(np.abs(lev - target_hpa)))
print(f"using lev[{ilev}] = {lev[ilev]:.2f} hPa")

fig3, ax3 = plt.subplots(figsize=(13, 3))
ax3.axhline(0.0, color="0.6", linewidth=1.0)
ax3.plot(dates_era5, U_th_era5[:, ilev], color="0.4", linewidth=1.3, label="ERA5")
ax3.plot(dates, U_th[:, ilev], color="#2a78d6", linewidth=1.5, label=f"CAM ({CASE})")
ax3.xaxis.set_major_locator(mdates.YearLocator(2))
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax3.set_ylabel("U (m s$^{-1}$)")
ax3.set_title(f"Equatorial U at {lev[ilev]:.1f} hPa")
ax3.legend(loc="lower left", ncol=2, frameon=False)
fig3.tight_layout()